# A Threat Model You Can Actually Run
### Companion notebook to *"A Systematic Threat Model for Runtime-Enforced Regulatory Compliance in Tokenised Real-World Asset Systems"*

Salaam, and welcome. Think of this notebook like sitting with your nani (grandmother)
on a Sunday afternoon while she explains something complicated using nothing but
things from around the house — chai, mangoes, the family shop, a wedding guest list.
That is exactly what we are going to do here, except the "something complicated" is
a research paper about keeping blockchain systems honest.

We are not going to just *read* the paper's claims. We are going to *build tiny toy
versions* of the systems it describes, break them the way the paper's attack trees
say they can be broken, and then patch them back up with the mitigations the paper
recommends. By the end, you won't just understand the paper — you'll have watched
it happen, in code, with your own eyes.

No fancy blockchain, no real money, no real risk. Just small honest Python classes,
pretending to be a big serious system, so we can poke at them safely.


## 1. What is "tokenised RWA compliance," beta?

Imagine your family owns a small shop. Instead of one person owning the whole
shop, you cut ownership into a hundred paper chits — "shares" — and hand them out
to relatives. Whoever holds a chit owns a piece of the shop and gets a piece of
the profit. That is, roughly, what "tokenising a real-world asset" means: taking
something real (a shop, a building, a bond) and representing ownership of it as
tokens on a computer ledger instead of paper chits.

Now here is the catch, and it's the whole reason this paper exists: not every
relative is *allowed* to hold a chit. Maybe the law says only people who have
registered with the government can hold a share. Maybe sanctioned people are
banned entirely. Someone has to *check* that every single chit-holder is actually
allowed to hold one — every time a chit changes hands, forever.

The paper's central question is simply: **where should that checking happen, and
what can go wrong with it?** Two answers exist:

- **Checked at the shop counter** (the "application layer" / smart contract): the
  shopkeeper checks IDs before handing over a chit. But if a customer finds a side
  door that doesn't go through the counter, the check never happens.
- **Checked by the chit itself, wired into the paper** (the "runtime layer"): no
  side door exists, because *every* chit transfer — through the front counter or
  otherwise — physically requires the check to pass first.

The paper studies systems built the second way, because that is the direction
serious institutional designs (Polymesh, Auer's "embedded supervision," and
similar proposals) are moving in. And its argument is: just because the side door
is welded shut doesn't mean the house is now safe. It just means the burglars have
to try something smarter than the side door.


## 2. The five rooms in the house

The paper abstracts every one of these systems into five pieces, like five rooms
in the same house:

| Room (module) | What it's like |
|---|---|
| **Identity & Credential Verification** | The guard at the gate, checking everyone's invitation card |
| **Compliance Parameter Store** | The rulebook on the shelf — "who is allowed in, this week" |
| **Compliance Gateway** | The single door everyone must walk through — no side doors |
| **Compliance Trail** | The guest-book diary, written once and never erased |
| **Governance / Quorum Authority** | The family elders' council, the only ones allowed to rewrite the rulebook |

Let's build all five as tiny Python classes. None of this is "real" cryptography —
it is a toy model, exactly detailed enough to let the paper's attack trees run.


In [ ]:

# Chalo, let's build our little house, room by room.

class CredentialIssuer:
    '''
    Think of this like the man at the mandi (market) with the official stamp.
    He stamps ration cards. If people trust his stamp, they trust the card.
    If somebody bribes HIM, every card he stamps after that is now suspicious —
    even though the stamp still LOOKS perfectly real.
    '''
    def __init__(self, name, trustworthy=True):
        self.name = name
        self.trustworthy = trustworthy  # a compromised issuer sets this False

    def issue_credential(self, participant_id, attribute):
        # "attribute" is something like 'accredited_investor' or 'sanctioned'.
        # A dishonest issuer will happily stamp a card for ANY attribute,
        # true or not -- exactly like a bribed clerk stamping any form you hand him.
        return {
            "issuer": self.name,
            "participant_id": participant_id,
            "attribute": attribute,
            "genuinely_entitled": self.trustworthy,  # only WE (the storyteller) know if this is a lie
        }


class VerificationModule:
    '''
    The guard at the gate. He doesn't personally know every guest --
    he just checks: "Is this card stamped by someone on MY trusted-stamps list?"
    If yes, and the card says the right thing, he waves you through.
    He has no way of knowing if the stamp-giver was bribed. That is not his job.
    '''
    def __init__(self, trusted_issuers):
        self.trusted_issuers = {i.name for i in trusted_issuers}

    def verify(self, credential, required_attribute):
        issuer_is_trusted = credential["issuer"] in self.trusted_issuers
        attribute_matches = credential["attribute"] == required_attribute
        return issuer_is_trusted and attribute_matches


class ComplianceParameterStore:
    '''
    The rulebook sitting on the shelf. It just says, in plain writing,
    "to move THIS kind of asset, you need to show THIS kind of card."
    Nobody is allowed to scribble in this rulebook except through the
    Governance Authority's front door -- at least, that's the plan.
    '''
    def __init__(self):
        self.rules = {}   # asset_class -> required_attribute
        self.version = 0

    def get_rule(self, asset_class):
        return self.rules.get(asset_class)

    def apply_update(self, asset_class, required_attribute):
        self.rules[asset_class] = required_attribute
        self.version += 1


class ComplianceTrail:
    '''
    The guest-book diary at the door. Every single person who walks past
    the guard gets written down -- whether they were let in or turned away --
    so that later, if anyone asks "who came to this wedding," there's an
    honest answer sitting in the diary, in ink, not pencil.
    '''
    def __init__(self, granularity="fine"):
        self.entries = []
        self.granularity = granularity  # "fine" writes exact sub-rule, "coarse" writes just the class

    def log(self, participant_id, asset_class, required_attribute, outcome):
        detail = required_attribute if self.granularity == "fine" else "eligibility-class-only"
        self.entries.append({
            "participant_id": participant_id,
            "asset_class": asset_class,
            "detail_logged": detail,
            "outcome": outcome,
        })


class ComplianceGateway:
    '''
    THE single door. This is the whole point of "runtime enforcement":
    there is supposed to be no window, no back door, no crawling under the
    fence -- every transfer walks through THIS door, and this door always
    asks the guard (VerificationModule) and checks the rulebook (Store)
    before letting anyone through.

    We give it a `mediation_enabled` switch so we can pretend, just for a
    moment, what happens if that promise is broken.
    '''
    def __init__(self, store, verification_module, trail, mediation_enabled=True, has_bug=False):
        self.store = store
        self.verification_module = verification_module
        self.trail = trail
        self.mediation_enabled = mediation_enabled  # False = a side door exists (Attack A1)
        self.has_bug = has_bug                       # True = the door's lock is broken (Attack A2)

    def process_transaction(self, participant_id, credential, asset_class):
        if not self.mediation_enabled:
            # There's a side door. Nobody even asked the guard anything.
            self.trail.log(participant_id, asset_class, "NO CHECK PERFORMED", "ACCEPTED (bypassed gateway)")
            return True

        required_attribute = self.store.get_rule(asset_class)

        if self.has_bug:
            # The lock is broken -- it clicks open no matter what you show it.
            self.trail.log(participant_id, asset_class, required_attribute, "ACCEPTED (gateway bug)")
            return True

        allowed = self.verification_module.verify(credential, required_attribute)
        outcome = "ACCEPTED" if allowed else "REJECTED"
        self.trail.log(participant_id, asset_class, required_attribute, outcome)
        return allowed


In [ ]:

# A quick, boring, HONEST run -- just to see our little house working
# the way it's supposed to, before anyone tries anything naughty.

good_issuer = CredentialIssuer("TrustedIssuerCo", trustworthy=True)
verification = VerificationModule(trusted_issuers=[good_issuer])
store = ComplianceParameterStore()
store.apply_update("security_token_class_A", required_attribute="accredited_investor")
trail = ComplianceTrail(granularity="fine")
gateway = ComplianceGateway(store, verification, trail)

honest_credential = good_issuer.issue_credential("participant_1", "accredited_investor")
print("Honest participant, correct card:", gateway.process_transaction("participant_1", honest_credential, "security_token_class_A"))

wrong_credential = good_issuer.issue_credential("participant_2", "just_a_regular_customer")
print("Honest participant, WRONG card:  ", gateway.process_transaction("participant_2", wrong_credential, "security_token_class_A"))

print("\nWhat's written in the diary so far:")
for entry in trail.entries:
    print(" -", entry)


## 3. Six kinds of naughty kids at the wedding

Every big wedding has a guard at the gate, and every big wedding, without fail,
has SOME kid trying to sneak in without an invite. The paper says: don't just
imagine one type of naughty kid. There are six very different types, and each
one causes a very different kind of trouble.

1. **Adv1 — no invite at all.** Just a stranger off the street, hoping nobody checks.
2. **Adv2 — has an invite, but for the WRONG event.** He's allowed at the mehndi, not the walima.
3. **Adv3 — is the printer who prints the invitation cards.** If he's bribed, he prints fake invites that look completely real.
4. **Adv4 — delivers the news to the caterer about the guest list.** If HE lies, the caterer (who trusts him) prepares food for people who shouldn't be there.
5. **Adv5 — a few of the family elders, teaming up.** If enough elders agree, they CAN legally rewrite the guest list themselves — that's not cheating, that's just how many elders it takes.
6. **Adv6 — the electrician who wired the whole wedding hall.** Doesn't own the guest list, but knows exactly where the wires are.

Nobody here needs to break any locks with brute force. They all just need to find
the crack that matches WHO they are.


In [ ]:

ADVERSARIES = {
    "Adv1": "Unauthenticated stranger -- no invite, no credential, just network access to the gate.",
    "Adv2": "Authenticated but ineligible guest -- has a real card, but not the right one for THIS door.",
    "Adv3": "Compromised credential issuer -- the man with the official stamp has been bribed.",
    "Adv4": "Compromised regulatory-data feed -- whoever is meant to update the rulebook is being fed lies.",
    "Adv5": "Colluding governance quorum -- enough family elders teaming up to legally rewrite the rulebook.",
    "Adv6": "Insider with protocol-level access -- the electrician who wired the hall, but isn't on the guest committee.",
}

for code_name, description in ADVERSARIES.items():
    print(f"{code_name}: {description}")


## 4. STRIDE, or: six flavours of "something went wrong"

STRIDE is just a checklist, invented long before blockchains existed, for the
six basic FLAVOURS a security problem can come in. Think of it like the six
flavours of chaat at a stall -- sweet, sour, spicy and so on -- except here
the "flavours" are:

- **S**poofing — pretending to be someone you're not
- **T**ampering — quietly changing something that shouldn't change
- **R**epudiation — being able to deny you ever did it
- **I**nformation disclosure — leaking something private
- **D**enial of service — jamming the whole system up
- **E**levation of privilege — giving yourself more power than you were ever handed

The paper checks each of the five rooms in our house against this list.
Here's Table 1 from the paper, reproduced as a small table.


In [ ]:

import pandas as pd

table1 = pd.DataFrame([
    {"Module": "Identity & Credential Verification", "Dominant STRIDE": "Spoofing, Information disclosure",
     "What it looks like": "A fake or revoked credential is waved through; leaked metadata shrinks the crowd of 'who could this be' down to a suspiciously small guess."},
    {"Module": "Compliance Parameter Store", "Dominant STRIDE": "Tampering, Elevation of privilege",
     "What it looks like": "A rule change lands in the rulebook that the elders' council never actually approved."},
    {"Module": "Compliance Gateway", "Dominant STRIDE": "Tampering, Denial of service",
     "What it looks like": "A bug lets a bad transfer slip through; or the door gets jammed with so many people it stops working."},
    {"Module": "Compliance Trail", "Dominant STRIDE": "Repudiation, Tampering",
     "What it looks like": "Pages get quietly torn out of the guest-book diary, or rewritten after the fact."},
    {"Module": "Governance / Quorum Authority", "Dominant STRIDE": "Elevation of privilege, Repudiation",
     "What it looks like": "A coalition of elders pushes through a rule change nobody else agreed to."},
    {"Module": "Oracle / Regulatory Data Feed", "Dominant STRIDE": "Spoofing, Tampering",
     "What it looks like": "Fake sanctions-list data is fed to the elders, who trust it because it LOOKS like the real government feed."},
])
table1


## 5. Attack tree 1 — sneaking a bad guest past the door

Right, now the fun part. There are exactly THREE ways, the paper says, to get a
transfer accepted that should have been rejected -- and NONE of them need you
to be a genius cryptographer. Let's do each one, in code.

**Branch A: don't go through the door at all.**


In [ ]:

# Attack A1 -- the side door. Somebody found a way to talk to the ledger
# directly, completely skipping the guard. This is EXACTLY like a wedding
# where someone climbs over the back wall instead of walking past the gate.

leaky_store = ComplianceParameterStore()
leaky_store.apply_update("security_token_class_A", required_attribute="accredited_investor")
leaky_trail = ComplianceTrail()
leaky_gateway = ComplianceGateway(leaky_store, verification, leaky_trail, mediation_enabled=False)  # <-- side door exists!

sneaky_credential = {"issuer": "nobody", "participant_id": "gatecrasher", "attribute": "made_this_up"}
result = leaky_gateway.process_transaction("gatecrasher", sneaky_credential, "security_token_class_A")
print("Did the gatecrasher get through the side door?", result)
print("Diary entry:", leaky_trail.entries[-1])

print("\nMITIGATION (M1 -- protocol-level interception): weld the side door shut.")
sealed_gateway = ComplianceGateway(leaky_store, verification, leaky_trail, mediation_enabled=True)
result = sealed_gateway.process_transaction("gatecrasher", sneaky_credential, "security_token_class_A")
print("Same gatecrasher, side door now welded shut:", result)


In [ ]:

# Attack A2 -- the door's lock is just... broken. A software bug means it
# clicks open for anyone, no matter what card they show. This is different
# from a side door: the guard IS standing there, he's just not doing his job.

buggy_gateway = ComplianceGateway(leaky_store, verification, leaky_trail, has_bug=True)
result = buggy_gateway.process_transaction("gatecrasher", sneaky_credential, "security_token_class_A")
print("Buggy gateway lets the gatecrasher through:", result)

print("\nMITIGATION (M2 -- formal verification + audit of the gateway's own code):")
fixed_gateway = ComplianceGateway(leaky_store, verification, leaky_trail, has_bug=False)
result = fixed_gateway.process_transaction("gatecrasher", sneaky_credential, "security_token_class_A")
print("Same gatecrasher, bug fixed and audited:", result)


**Branch B: don't sneak past the guard -- rewrite the rulebook instead.**

This is the one the paper spends the most time on, because it has a genuinely
clever twist. To rewrite the rulebook you need enough family elders to agree
(a "threshold" -- say, 3 out of 5). But here is the twist: once you HAVE 3 out
of 5 elders agreeing, there is nothing stopping them, mathematically speaking.
The other 2 elders don't get a vote to block it after the fact -- UNLESS the
family has a SEPARATE house rule, like "any rule change has to sit for a week
before it counts, so the other elders have time to object."

That waiting week is not part of the signature-counting at all. It's a totally
separate safeguard. Remove it, and the moment 3 elders agree, the rule changes
-- instantly, with nobody able to stop it.


In [ ]:

class GovernanceAuthority:
    '''
    The council of family elders. `threshold` elders (out of `quorum_size`)
    must agree before a new rule counts. `challenge_period_active` is our
    separate house rule -- a cooling-off week where anyone can still object.
    '''
    def __init__(self, quorum_size, threshold, challenge_period_active=True):
        self.quorum_size = quorum_size
        self.threshold = threshold
        self.challenge_period_active = challenge_period_active

    def propose_update(self, colluding_elders, store, asset_class, new_required_attribute):
        # Step B1a: do the colluding elders even meet the threshold?
        if len(colluding_elders) < self.threshold:
            return "REJECTED -- not enough elders agreed (below threshold)"

        # Step B1b: is there a cooling-off week for the OTHER elders to object?
        if self.challenge_period_active:
            return ("BLOCKED FOR NOW -- threshold met, but the challenge/review "
                    "window means this has to sit and wait; it can still be caught.")

        # No cooling-off week exists. Threshold met = done. Rule changes immediately.
        store.apply_update(asset_class, new_required_attribute)
        return f"RULE CHANGED IMMEDIATELY to '{new_required_attribute}' -- no safeguard stopped it."


council_with_safeguard = GovernanceAuthority(quorum_size=5, threshold=3, challenge_period_active=True)
council_without_safeguard = GovernanceAuthority(quorum_size=5, threshold=3, challenge_period_active=False)

colluding_elders = ["elder_1", "elder_2", "elder_3"]  # exactly meets the threshold of 3

print("WITH the challenge period (mitigation M3 switched ON):")
print(" ", council_with_safeguard.propose_update(colluding_elders, store, "security_token_class_A", "anyone_at_all"))

print("\nWITHOUT the challenge period (mitigation M3 switched OFF):")
print(" ", council_without_safeguard.propose_update(colluding_elders, store, "security_token_class_A", "anyone_at_all"))
print(" New rule in the store now:", store.get_rule("security_token_class_A"))


In [ ]:

# Attack B2 -- a different, smaller trick: instead of taking over the whole
# council, just exploit the LAG between "a new law was announced" and
# "our rulebook actually got updated to reflect it." Nobody colluded --
# the elders were simply slow.

class LatencyWindowDemo:
    '''
    Ammi hears on the news that a certain family is now banned from buying
    mangoes. She tells the shop clerk immediately -- but updating the big
    chalkboard behind the counter takes a few minutes. During those few
    minutes, the banned family can still walk up and buy mangoes, because
    the chalkboard hasn't caught up to the news yet.
    '''
    def __init__(self, emergency_tier_enabled=False):
        self.emergency_tier_enabled = emergency_tier_enabled
        # without the emergency tier, updating the board takes 10 "ticks";
        # with it, an urgent update is fast-tracked in 1 "tick".
        self.update_delay_ticks = 1 if emergency_tier_enabled else 10

print("MITIGATION (M4 -- expedited emergency rule-activation tier):")
slow = LatencyWindowDemo(emergency_tier_enabled=False)
fast = LatencyWindowDemo(emergency_tier_enabled=True)
print(f"  Without the fast lane: the banned family has {slow.update_delay_ticks} ticks to sneak a purchase in.")
print(f"  With the fast lane:    the banned family only has {fast.update_delay_ticks} tick  to sneak a purchase in.")


**Branch C: don't touch the rulebook or the door -- forge the invitation card itself.**

Remember the man at the mandi with the official stamp? If HE is dishonest,
or if the guard trusts stamps from every two-bit stall in the bazaar instead of
only the two or three stalls the family actually recognises, then a fake card
sails right through -- because as far as the guard can tell, it's a real stamp.


In [ ]:

sketchy_issuer = CredentialIssuer("RandomBazaarStallXYZ", trustworthy=False)

# our original guard, `verification`, only trusts `good_issuer` -- so this should fail:
fake_credential = sketchy_issuer.issue_credential("sneaky_participant", "accredited_investor")
print("Fake card from an issuer NOT on the guard's trust list:",
      verification.verify(fake_credential, "accredited_investor"))

print("\nMITIGATION (M5 -- issuer allow-list): the guard simply never heard of this stall.")
print("If instead the guard's trust list carelessly included this stall too:")
careless_guard = VerificationModule(trusted_issuers=[good_issuer, sketchy_issuer])
print(" ", careless_guard.verify(fake_credential, "accredited_investor"), "-- and now the fake card walks right through.")


## 6. Attack tree 2 — figuring out WHO someone is, without breaking any lock

This one doesn't try to sneak anyone past the guard at all. Instead, it tries
to work out a real person's identity just by watching PATTERNS -- like a nosy
neighbour who never opens your mail, but still knows exactly when the courier
visits, because she's always at her window.


In [ ]:

# Attack D1 -- timing correlation. If a withdrawal from an exchange happens
# at 3:14pm for exactly 500 units, and a deposit on-chain happens at 3:14pm
# for exactly 500 units, that's not proof -- but it's a VERY strong hint,
# the same way seeing your neighbour's light switch on the instant the
# courier's bike pulls up outside is a very strong hint.

off_chain_withdrawals = [("exchange_customer_A", "15:14:02", 500), ("exchange_customer_B", "09:03:10", 12)]
on_chain_deposits =     [("wallet_X9f2", "15:14:05", 500),          ("wallet_A001", "18:40:00", 7)]

print("MITIGATION OFF (no batching/delay) -- matching timestamps + amounts:")
for who, t_off, amt_off in off_chain_withdrawals:
    for wallet, t_on, amt_on in on_chain_deposits:
        if amt_off == amt_on:
            print(f"  '{who}' (withdrew at {t_off}) very likely == '{wallet}' (deposited at {t_on}) -- amounts match exactly.")

print("\nMITIGATION ON (M7 -- batched/delayed submission): if wallet_X9f2's deposit")
print("were held back and released at a random later time mixed in with 50 other")
print("deposits, the '500 units at 15:14' match above simply would not exist to find.")


In [ ]:

# Attack D2 -- the diary writes down TOO MUCH detail. Compare a coarse
# diary entry ("someone from the accredited-investor group came by") to a
# fine-grained one ("someone whose exact sub-rule was 'accredited AND
# resident-of-Sharjah AND holds-more-than-10-tokens' came by") -- the second
# one narrows down "who could this possibly be" to almost nobody.

fine_trail = ComplianceTrail(granularity="fine")
coarse_trail = ComplianceTrail(granularity="coarse")

fine_trail.log("participant_9", "security_token_class_A", "accredited AND resident_of_Sharjah AND holds>10", "ACCEPTED")
coarse_trail.log("participant_9", "security_token_class_A", "accredited AND resident_of_Sharjah AND holds>10", "ACCEPTED")

print("Fine-grained diary entry (leaks a LOT):  ", fine_trail.entries[-1])
print("Coarse diary entry (M6 -- leaks much less):", coarse_trail.entries[-1])


In [ ]:

# Attack D3 -- this one needs TWO things at once (an AND, not an OR):
# the bribed stamp-man's private paperwork, AND a timing match between
# when he stamped the card and when that card was used. Only having ONE
# of the two tells you much less -- you need BOTH to be confident.

issuer_side_logs = {"credential_id_882": {"real_name": "actual_person_Z", "stamped_at": "10:00:00"}}
onchain_proof_submission_time = "10:00:04"  # four seconds after the stamp -- suspiciously close

got_issuer_logs = "credential_id_882" in issuer_side_logs          # D3a
timestamps_line_up = onchain_proof_submission_time.startswith("10:00:0")  # D3b (crude, but you get the idea)

print("D3a (leaked issuer paperwork)          :", got_issuer_logs)
print("D3b (issuance time matches usage time) :", timestamps_line_up)
print("D3 succeeds only if BOTH are true       :", got_issuer_logs and timestamps_line_up)
print("\nMITIGATION (M5, reused as issuer data-minimisation): if the issuer never")
print("kept 'real_name' next to 'credential_id' in the first place, D3a fails on its own,")
print("and the AND-condition can never be satisfied, no matter what D3b finds.")


## 7. Attack tree 3 — when two houses don't talk to each other fast enough

Everything so far happened inside ONE house. This last attack tree only shows
up once you have TWO houses (two chains) connected by a bridge -- say, one shop
in Dubai and its sister shop in Abu Dhabi, linked by a single phone line.


In [ ]:

# Attack E1 -- the propagation lag. Dubai hears the new rule instantly.
# Abu Dhabi is still running the OLD rule until the phone call gets through.

dubai_store = ComplianceParameterStore()
abu_dhabi_store = ComplianceParameterStore()
dubai_store.apply_update("bond_class_9", required_attribute="accredited_investor")
abu_dhabi_store.apply_update("bond_class_9", required_attribute="accredited_investor")  # both start in sync

print("Dubai just heard: this bond now ALSO requires 'sanctions_clear'.")
dubai_store.apply_update("bond_class_9", required_attribute="sanctions_clear")
print("Dubai's rule now: ", dubai_store.get_rule("bond_class_9"))
print("Abu Dhabi's rule still:", abu_dhabi_store.get_rule("bond_class_9"), " <-- the phone call hasn't arrived yet")

print("\nA transfer racing through Abu Dhabi RIGHT NOW is still checked against the OLD rule.")
print("MITIGATION (M9 -- flag the pending change to relayers ahead of time) shrinks this window.")

print("\n...a few ticks later, the phone call finally arrives:")
abu_dhabi_store.apply_update("bond_class_9", required_attribute="sanctions_clear")
print("Abu Dhabi's rule now:  ", abu_dhabi_store.get_rule("bond_class_9"), " <-- caught up, window closed")


In [ ]:

# Attack E2 -- Dubai and Abu Dhabi simply DISAGREE about the rule, full stop.
# Not because of a delay -- because the family never actually agreed on ONE
# rule for this bond in the first place. There is no clever code fix for
# "two shops that never agreed on the same price list."

dubai_classification = "unrestricted"
abu_dhabi_classification = "requires_accreditation"

cheapest_route = "Dubai" if dubai_classification == "unrestricted" else "Abu Dhabi"
print(f"The same bond is '{dubai_classification}' in Dubai but '{abu_dhabi_classification}' in Abu Dhabi.")
print(f"A shrewd trader simply routes the transfer through {cheapest_route} -- whichever shop is looser.")
print("\nNo mitigation toggle exists for this one. Fixing it means the FAMILY has to sit down")
print("and agree on one classification -- a governance/legal conversation, not a code patch.")


In [ ]:

# Attack E3 -- an AND again: E3a (a real, valid card from Dubai's OWN guard)
# PLUS E3b (Abu Dhabi's guard trusting the bridge's word for it, instead of
# checking the card again himself).

class Bridge:
    '''
    The phone line between Dubai and Abu Dhabi. It faithfully tells Abu Dhabi
    WHAT was transferred -- but by default it does NOT tell Abu Dhabi whether
    the person receiving it was ever actually checked against ABU DHABI's own
    rules. It assumes "Dubai already checked, so it must be fine."
    '''
    def __init__(self, destination_reverifies=False):
        self.destination_reverifies = destination_reverifies  # M8

    def bridge_transfer(self, origin_credential_was_valid, destination_gateway, participant_id, asset_class):
        if not origin_credential_was_valid:
            return "REJECTED -- didn't even pass the origin chain's own guard (E3a failed)"

        if not self.destination_reverifies:
            # E3b: Abu Dhabi just takes Dubai's word for it. No second check.
            return "ACCEPTED on destination chain -- asset state trusted blindly, eligibility never re-checked"

        # M8 switched on: Abu Dhabi insists on checking its OWN rulebook, from scratch.
        required_attribute = destination_gateway.store.get_rule(asset_class)
        return f"Destination re-checks independently against its own rule ('{required_attribute}') before accepting"


trusting_bridge = Bridge(destination_reverifies=False)
careful_bridge = Bridge(destination_reverifies=True)

print("E3a passed (valid credential shown on origin chain), E3b open (no re-check):")
print(" ", trusting_bridge.bridge_transfer(True, sealed_gateway, "participant_Z", "bond_class_9"))

print("\nMITIGATION (M8 -- cross-chain compliance-state attestation), E3b now closed:")
print(" ", careful_bridge.bridge_transfer(True, sealed_gateway, "participant_Z", "bond_class_9"))


## 8. So... where does the real danger actually live?

Here is Table 2 from the paper -- the likelihood, impact, and *residual* risk
(the risk that's STILL left, even after every mitigation above is switched on)
for each scenario we just played with.


In [ ]:

table2 = pd.DataFrame([
    {"Scenario": "A1/A2 -- raw tx / gateway bug",              "Likelihood": "Low",        "Impact": "High",   "Residual risk": "Low"},
    {"Scenario": "B1 -- governance quorum collusion",           "Likelihood": "Low-Medium", "Impact": "High",   "Residual risk": "Medium"},
    {"Scenario": "B2 -- governance latency-window exploit",     "Likelihood": "Medium",     "Impact": "Medium", "Residual risk": "Low-Medium"},
    {"Scenario": "C1 -- unauthorised/compromised issuer",       "Likelihood": "Low-Medium", "Impact": "High",   "Residual risk": "Medium"},
    {"Scenario": "D1/D2 -- metadata / trail-granularity leak",  "Likelihood": "Medium-High","Impact": "Medium", "Residual risk": "Medium"},
    {"Scenario": "D3 -- issuer collusion for deanonymisation",  "Likelihood": "Low",        "Impact": "Medium", "Residual risk": "Low-Medium"},
    {"Scenario": "E1/E2 -- cross-chain lag / misclassification","Likelihood": "Medium",     "Impact": "Medium", "Residual risk": "Medium"},
    {"Scenario": "E3 -- bridge identity discontinuity",         "Likelihood": "Medium",     "Impact": "High",   "Residual risk": "Medium-High"},
])
table2


## 9. The moral of the story

Notice something, beta: not ONE of the scenarios we just ran needed anybody to
break a lock, crack an encryption, or out-smart the mathematics. The welded-shut
side door (Branch A) really did stay shut -- that part of the promise held up.

Every single time the risk stayed stubbornly "Medium" or higher even AFTER the
mitigation was switched on, it was because the crack wasn't in the code at all.
It was in a PERSON or a GROUP OF PEOPLE: the elders who could collude, the
stamp-man who could be bribed, the two shops that never agreed on one price list.
No amount of formal verification fixes an uncle who's decided to cheat.

That is the paper's real finding, dressed up in plain clothes: **when you weld
the technical side door shut, the danger doesn't disappear -- it just moves next
door, into governance and trust.** Auditing the code is necessary. It has never
been sufficient.

And that, chalo, is the whole paper -- explained the way your nani would explain
it, with mangoes and elders and a guard at the gate, and not one line of it
untrue to what the paper actually says.
